# NC-TCN vs NC-SSM Ablation Study
**NC-SSM-20K** (SSM scan, sequential) vs **NC-TCN-20K** (Dilated Conv1D, parallel)

Same NC frontend (STFT → SNR → SpectralGate → DualPCEN → InstanceNorm)  
Backend only: SSM scan → Dilated Causal Conv1D (3 layers, dilation 1,2,4)

| Model | Params | Python Speed | MCU Speed | INT8 |
|-------|--------|-------------|-----------|------|
| NC-SSM-20K | 20,044 | 54ms | ~13ms | Risky |
| **NC-TCN-20K** | **21,689** | **17ms** | **~8ms** | **Stable** |

In [ ]:
# 1. GitHub에서 프로젝트 clone
import os

dst = '/content/NanoMamba-Interspeech2026'
if not os.path.exists(dst):
    !git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP2026.git {dst}
    print('Clone done!')
else:
    os.chdir(dst)
    !git pull
    print('Updated!')

os.chdir(dst)
print(f'Working dir: {os.getcwd()}')

In [ ]:
!pip install -q torchaudio sounddevice

In [ ]:
# 3. NC-TCN 모델 검증 + GPU 속도 비교
import sys
sys.path.insert(0, '/content/NanoMamba-Interspeech2026')

from nanomamba import create_nc_tcn_20k, create_nanomamba_nc_20k
import torch, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tcn = create_nc_tcn_20k().to(device).eval()
ssm = create_nanomamba_nc_20k().to(device).eval()

p_tcn = sum(p.numel() for p in tcn.parameters())
p_ssm = sum(p.numel() for p in ssm.parameters())
print(f'\nNC-TCN-20K:  {p_tcn:>7,} params ({p_tcn*4/1024:.1f} KB)')
print(f'NC-SSM-20K:  {p_ssm:>7,} params ({p_ssm*4/1024:.1f} KB)')
print(f'Param ratio: {p_tcn/p_ssm:.2f}x')

# Speed benchmark
audio = torch.randn(1, 16000, device=device)
with torch.no_grad():
    tcn(audio); ssm(audio)  # warmup
    if device == 'cuda': torch.cuda.synchronize()

N = 100
if device == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N):
    with torch.no_grad(): tcn(audio)
if device == 'cuda': torch.cuda.synchronize()
tcn_ms = (time.perf_counter() - t0) / N * 1000

if device == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N):
    with torch.no_grad(): ssm(audio)
if device == 'cuda': torch.cuda.synchronize()
ssm_ms = (time.perf_counter() - t0) / N * 1000

print(f'\n{"="*50}')
print(f'  GPU Inference (avg {N} runs)')
print(f'{"="*50}')
print(f'  NC-TCN-20K:  {tcn_ms:6.1f}ms  (Conv1D, parallel)')
print(f'  NC-SSM-20K:  {ssm_ms:6.1f}ms  (SSM scan, sequential)')
print(f'  Speedup:     {ssm_ms/tcn_ms:6.1f}x faster')
print(f'{"="*50}')

In [ ]:
# 4. 전체 NC-TCN 라인업 학습 (Tiny + Matched + 20K)
!python train_colab.py \
    --models "NC-TCN-Tiny,NC-TCN-Matched,NC-TCN-20K" \
    --epochs 30 \
    --batch_size 128 \
    --lr 3e-3 \
    --noise_aug \
    --noise_curriculum_v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full

In [ ]:
# 5. 노이즈 평가: NC-TCN vs NC-SSM vs DS-CNN (5 noise x 7 SNR)
!python train_colab.py \
    --models "NC-TCN-20K,NanoMamba-NC-20K,DS-CNN-S" \
    --eval_only \
    --noise_aug \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range -15,-10,-5,0,5,10,15

In [ ]:
# 6. 프로파일: MACs, Latency, Memory 비교
!python train_colab.py \
    --models "NC-TCN-20K,NanoMamba-NC-20K,DS-CNN-S" \
    --profile

In [ ]:
# 7. 체크포인트 Drive에 저장
from google.colab import drive
import shutil

drive.mount('/content/drive')

src = './checkpoints_full/NC-TCN-20K'
dst = '/content/drive/MyDrive/NC-SSM-checkpoints/NC-TCN-20K'
if os.path.exists(src):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Checkpoint saved: {dst}')
else:
    print('No checkpoint found - train first (Cell 4)')